In [1]:
%load_ext autoreload
%autoreload 2

## 1. Imports

In [2]:
import os
import sys

sys.path.append("..")

import random

import numpy as np
import torch
import torch.distributions as TD
import wandb
from tqdm import tqdm

from src.auxiliary_models.mlp_based import FullyConnectedMLP
from src.costs.lse import MLPLSECost
from src.costs.mlp_based import MLPCost, MLPL2Cost
from src.models.energy_based import EGEOT
from src.plotting.distributions import plot_swiss_roll
from src.plotting.parameters import plot_B_parameters
from src.potentials.mlp_based import MLPPotential
from src.samplers.energy_based.sample_buffer import SampleBufferEgEOT
from src.samplers.from_dataset import DatasetSampler
from src.samplers.primary import StandardNormalSampler, SwissRollSampler
from src.utils.discrete_ot import OTPlanSampler
from src.utils.paired import generate_paired_data, get_GT_points, get_paired_sampler
from src.utils.train import compute_loss, update_average

In [3]:
device = torch.device(f"cuda:{torch.cuda.current_device()}" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

In [4]:
torch.set_default_device(device)
dtype = torch.float64
torch.torch.set_default_dtype(dtype)

## 2. Config

In [5]:
from configs.energy_based.cost import MLPCostConfig, MLPLSECostConfig, MLPL2CostConfig
from configs.energy_based.dataset import DatasetConfig, MiniBatchConfig
from configs.energy_based.model import EBMConfig
from configs.energy_based.optimizer import OptPairedConfig, OptUnpairedConfig
from configs.energy_based.potential import PotentialConfig
from configs.energy_based.sampling import LangevinConfig
from configs.energy_based.train import TrainConfig

In [6]:
Q_X_UNPAIRED_SAMPLES = 1024
R_Y_UNPAIRED_SAMPLES = 1024
P_XY_PAIRED_SAMPLES = 1024
LR_PAIRED = 2e-4
LR_UNPAIRED = 2e-4
SAMPLING_NUM_ITER = 100
MAX_STEPS = 1000
COST_FUNCTION = "MLP"
PAIRED_BATCH_SIZE = 1024
UNPAIRED_BATCH_SIZE = 1024

# For MLPCost
HIDDEN_LAYERS = [128, 128]

# For MLPLSECost
M_POTENTIALS = 2
LOG_V_M_HIDDEN_CHANNELS = [128, 128]
B_M_HIDDEN_CHANNELS = [128, 128]

# For MLPL2Cost
X_HIDDEN_LAYERS: list[int] = [128, 128]
Y_HIDDEN_LAYERS: list[int] = [128, 128]

In [7]:
dataset_config = DatasetConfig(
    P_XY_paired=P_XY_PAIRED_SAMPLES, Q_X_unpaired=Q_X_UNPAIRED_SAMPLES, R_Y_unpaired=R_Y_UNPAIRED_SAMPLES
)
minibatch_config = MiniBatchConfig()

potential_config = PotentialConfig()
if COST_FUNCTION == "MLP":
    cost_config = MLPCostConfig(hidden_layers=HIDDEN_LAYERS)
elif COST_FUNCTION == "MLPLSE":
    cost_config = MLPLSECostConfig(
        m_potentials=M_POTENTIALS,
        log_v_m_hidden_channels=LOG_V_M_HIDDEN_CHANNELS,
        b_m_hidden_channels=B_M_HIDDEN_CHANNELS,
    )
elif COST_FUNCTION == "MLPL2":
    cost_config = MLPL2CostConfig(
        x_hidden_layers=X_HIDDEN_LAYERS,
        y_hidden_layers=Y_HIDDEN_LAYERS,
    )
else:
    raise ValueError(f"Unknown cost function: {COST_FUNCTION}!")
model_config = EBMConfig(sampling=LangevinConfig(num_iterations=SAMPLING_NUM_ITER))

opt_unpaired_config = OptUnpairedConfig(lr=LR_UNPAIRED)
opt_paired_config = OptPairedConfig(lr=LR_UNPAIRED)

train_config = TrainConfig(
    steps_to=MAX_STEPS, paired_batch_size=PAIRED_BATCH_SIZE, unpaired_batch_size=UNPAIRED_BATCH_SIZE
)

In [8]:
torch.manual_seed(train_config.seed)
np.random.seed(train_config.seed)
random.seed(train_config.seed)

## 3. Create data and samplers

In [9]:
X_sampler = StandardNormalSampler(dim=dataset_config.x_dim, device=device)
Y_sampler = SwissRollSampler(dim=dataset_config.y_dim, device=device, dtype=dtype)

In [10]:
otp_sampler = OTPlanSampler(**minibatch_config.model_dump())

In [11]:
data_dir = "checkpoints/Tensors"
file_postfix = f"{minibatch_config.cost_function}_{dataset_config.P_XY_paired}"

In [12]:
X_paired_train, Y_paired_train, X_paired_test, Y_paired_test = generate_paired_data(
    X_sampler, Y_sampler, otp_sampler, dataset_config.P_XY_paired, "./checkpoints/Tensors", file_postfix, device=device
)

/beegfs/home/m.persiyanov/Light-GCOT/src/utils/paired.py:48: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  X_paired_train = torch.load(os.path.join(save_dir, f"X_paired_trai

In [13]:
pd_train_sampler = get_paired_sampler(
    X_paired_train, Y_paired_train, train_config.paired_batch_size, dataset_config.P_XY_paired, device
)

In [14]:
X_unpaired_test = X_sampler.sample(dataset_config.P_XY_paired)
Y_unpaired_test = Y_sampler.sample(dataset_config.P_XY_paired)

In [15]:
if dataset_config.Q_X_unpaired > 0:
    source_data = X_sampler.sample(dataset_config.Q_X_unpaired)
    usd_sampler = DatasetSampler(source_data, device=device) # usd - unpaired source data
else:
    usd_sampler = DatasetSampler(X_paired_train, device=device)

if dataset_config.R_Y_unpaired > 0:
    target_data = Y_sampler.sample(dataset_config.R_Y_unpaired)
    utd_sampler = DatasetSampler(target_data, device=device) # utd - unpaired target data
else:
    utd_sampler = DatasetSampler(Y_paired_train, device=device)

## 4. Model initialization

In [16]:
potential = MLPPotential(**potential_config.model_dump())

In [17]:
if COST_FUNCTION == "MLP":
    cost = MLPCost(**cost_config.model_dump())
elif COST_FUNCTION == "MLPLSE":
    cost = MLPLSECost(**cost_config.model_dump())
elif COST_FUNCTION == "MLPL2":
    cost = MLPL2Cost(**cost_config.model_dump())
else:
    raise ValueError(f"Unknown cost function: {COST_FUNCTION}!")

In [18]:
# TODO: add to config
BASIC_NOISE_VAR = 1.0
P_SAMPLE_BUFFER_REPLAY = 0.95
SAMPLE_BUFFER_SAMPLES = 10000

In [19]:
basic_noise_gen = TD.Normal(torch.tensor([0.0, 0.0]).to(device), torch.tensor([1.0, 1.0]).to(device) * BASIC_NOISE_VAR)

sample_buffer_instance = SampleBufferEgEOT(
    basic_noise_gen, p=P_SAMPLE_BUFFER_REPLAY, max_samples=SAMPLE_BUFFER_SAMPLES, device=device
)

In [20]:
model = EGEOT(potential, cost, sample_buffer_instance, model_config)

In [21]:
# For EMA update
if train_config.ema_update:
    model_copy = EGEOT(potential, cost, sample_buffer_instance, model_config)

## 5. Optimizers initialization

In [22]:
D_opt_unpaired = torch.optim.Adam(model.potential.parameters(), **opt_unpaired_config.model_dump())

In [23]:
D_opt_paired = torch.optim.Adam(model.cost.parameters(), **opt_paired_config.model_dump())

In [24]:
# TODO: refactor this config
EXP_META_INFO = ""
EXP_NAME = (
    "EgEOT_Swiss_Roll_"
    + f"COST_FUNCTION_{COST_FUNCTION}_"
    + f"P_XY_PAIRED_{dataset_config.P_XY_paired}_"
    + f"Q_X_UNPAIRED_{dataset_config.Q_X_unpaired}_"
    + f"R_Y_UNPAIRED_{dataset_config.R_Y_unpaired}_"
    + f"LR_PAIRED_{opt_paired_config.lr}_"
    + f"LR_UNPAIRED_{opt_unpaired_config.lr}_"
    + f"MINIBATCH_COST_{minibatch_config.cost_function}_"
    + f"SAMPLING_STEPS_{model_config.sampling.num_iterations}_"
    + EXP_META_INFO
)
OUTPUT_PATH = "../checkpoints/{}".format(EXP_NAME)

config = dict(
    COST_FUNCTION=COST_FUNCTION,
    X_DIM=dataset_config.x_dim,
    Y_DIM=dataset_config.y_dim,
    D_LR_PAIRED=opt_paired_config.lr,
    D_LR_UNPAIRED=opt_unpaired_config.lr,
    BATCH_SIZE=train_config.unpaired_batch_size,
    P_XY_PAIRED_SAMPLES=dataset_config.P_XY_paired,
    Q_X_UNPAIRED_SAMPLES=dataset_config.Q_X_unpaired,
    R_Y_UNPAIRED_SAMPLES=dataset_config.R_Y_unpaired,
)

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH, exist_ok=True)

In [25]:
if train_config.steps_from > 0:
    D_opt_unpaired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{train_config.steps_from}.pt")))
    D_opt_paired.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_paired_{train_config.steps_from}.pt")))

## 6. Model training

In [26]:
starting_points = torch.tensor([[-2.0, 0.0], [0.0, 0.0], [0.0, -2.0]])
num_ending_points = 64

In [27]:
num_starting_points_paired = 5
indices = random.choices(range(dataset_config.P_XY_paired), k=num_starting_points_paired)
starting_points_paired = X_paired_train[indices]
ending_points_paired = Y_paired_train[indices]

In [28]:
gt_Y_points = get_GT_points(X_sampler, Y_sampler, otp_sampler, starting_points)

  0%|                                                                                                                                                                                  | 0/64 [00:00<?, ?it/s]/trinity/home/m.persiyanov/miniconda3/envs/text/lib/python3.11/site-packages/ot/bregman/_sinkhorn.py:531: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn("Sinkhorn did not converge. You might want to "
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 64/64 [00:00<00:00, 86.69it/s]


In [ ]:
wandb.init(name=EXP_NAME, config=config)

for step in tqdm(range(train_config.steps_from, train_config.steps_to)):
    # training loop
    D_opt_unpaired.zero_grad()

    X = usd_sampler.sample(train_config.unpaired_batch_size)
    Y = utd_sampler.sample(train_config.unpaired_batch_size)

    output_unpaired = model.compute_unpaired_loss(X, Y, compute_stats=True)
    D_loss_unpaired = output_unpaired["loss"]

    wandb.log({f"Unpaired: Loss": D_loss_unpaired.item()}, step=step)
    wandb.log({f"Unpaired: \int f(y)": output_unpaired["int_potential"].item()}, step=step)
    wandb.log({f"Unpaired: \int\log Z": output_unpaired["int_log_Z"].item()}, step=step)
    wandb.log({f"Unpaired: -E(x, y)": output_unpaired["neg_energy_t"].item()}, step=step)
    wandb.log({f"Unpaired: c(x, y)": output_unpaired["cost_t"].item()}, step=step)
    wandb.log({f"Unpaired: f(y)": output_unpaired["potential_t"].item()}, step=step)
    wandb.log({f"Unpaired: noise": output_unpaired["noise"].item()}, step=step)

    D_opt_paired.zero_grad()
    X_paired, Y_paired = pd_train_sampler.sample(train_config.paired_batch_size)

    output_paired = model.compute_paired_loss(X_paired, Y_paired, compute_stats=True)
    D_loss_paired = output_paired["loss"]

    wandb.log({f"Paired: Loss": D_loss_paired.item()}, step=step)
    # wandb.log({f"Paired: -E(x, y)": output_paired["neg_energy_t"].item()}, step=step)
    # wandb.log({f"Paired: c(x, y)": output_paired["cost_t"].item()}, step=step)
    # wandb.log({f"Paired: f(y)": output_paired["potential_t"].item()}, step=step)
    # wandb.log({f"Paired: noise": output_paired["noise"].item()}, step=step)

    D_loss = D_loss_unpaired + D_loss_paired
    D_loss.backward()
    D_opt_paired.step()
    D_opt_unpaired.step()

    if train_config.ema_update:
        update_average(model_copy, model, 0.99)
        model = model_copy
    else:
        model = model

    wandb.log({f"Loss": D_loss}, step=step)
    wandb.log(
        {f"Train paired loss": compute_loss(model, X_paired_train, Y_paired_train, X_paired_train, Y_paired_train)},
        step=step,
    )
    wandb.log(
        {f"Test paired loss": compute_loss(model, X_paired_test, Y_paired_test, X_paired_test, Y_paired_test)},
        step=step,
    )
    wandb.log(
        {f"Test unpaired loss": compute_loss(model, X_unpaired_test, Y_unpaired_test, X_paired_test, Y_paired_test)},
        step=step,
    )

    if step % train_config.plot_every == 0:
        distr_dict = plot_swiss_roll(
            {"EBM": model},
            X_sampler,
            Y_sampler,
            X_paired_train,
            Y_paired_train,
            starting_points,
            gt_Y_points,
            log=True,
        )
        if COST_FUNCTION == "MLPLSE":
            B_dict = B_dict = plot_B_parameters(model.cost, starting_points, log=True)
            distr_dict = distr_dict | B_dict
        wandb.log(distr_dict)
        torch.save(model.potential.state_dict(), os.path.join(OUTPUT_PATH, f"potential_{step}.pt"))

torch.save(model.potential.state_dict(), os.path.join(OUTPUT_PATH, f"D_{train_config.steps_to}.pt"))
torch.save(D_opt_paired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_paired_{train_config.steps_to}.pt"))
torch.save(D_opt_unpaired.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_unpaired_{train_config.steps_to}.pt"))

wandb.finish()

wandb: Currently logged in as: muxaujl11110. Use `wandb login --relogin` to force relogin


  0%|                                                                                                                                                                                | 0/1000 [00:00<?, ?it/s]/trinity/home/m.persiyanov/miniconda3/envs/text/lib/python3.11/site-packages/torch/autograd/graph.py:768: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /opt/conda/conda-bld/pytorch_1720538435607/work/aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
 57%|███████████████████████████████████████████████████████████████████████████████████████████████▎                                                                      | 574/1000 [12:18<09:09,  1.29s/it]

## Plotting

In [ ]:
plot_swiss_roll(
    {"EBM": model},
    X_sampler,
    Y_sampler,
    X_paired_train,
    Y_paired_train,
    starting_points,
    gt_Y_points,
) 